In [ ]:
import os
import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from dotenv import load_dotenv

load_dotenv() # This looks for a file named .env in your folder

# 1. Setup Authentication via Environment Variable
key_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
if not key_path:
    raise ValueError("Missing GOOGLE_APPLICATION_CREDENTIALS environment variable.")

scopes = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds = Credentials.from_service_account_file(key_path, scopes=scopes)
client = gspread.authorize(creds)

# 2. Fetch Data as "List of Lists"
# This avoids the gspread header validation error
spreadsheet_id = os.environ.get('GOOGLE_SHEET_ID')
sheet = client.open_by_key(spreadsheet_id).sheet1
all_values = sheet.get_all_values()

# 3. Load into Pandas
# We take row 0 as headers and the rest as data
df = pd.DataFrame(all_values[1:], columns=all_values[0])

# 4. Clean the Headers (The "Pandas Magic")
# This handles the empty string headers you encountered
df.columns = [f"Unnamed_{i}" if col.strip() == "" else col for i, col in enumerate(df.columns)]

# If there are still duplicates (e.g., two columns named 'Status'), 
# this line ensures they are unique (e.g., 'Status', 'Status.1')
df.columns = pd.Index([f"{c}.{i}" if df.columns.tolist().count(c) > 1 else c 
                       for i, c in enumerate(df.columns)])

# 5. Final Output for Dashboard
# Most frontend dashboards expect a list of dictionaries
dashboard_data = df.to_dict(orient='records')

print(f"Successfully processed {len(df)} rows.")
print("Columns found:", df.columns.tolist())

Successfully processed 900 rows.
Columns found: ['Days in Warehouse', 'Manufacturer Name', 'Category', 'General', 'Name', 'Lot Number', 'Notes', 'Quantity', 'Date of Expiration', 'Pallet Number', 'Box Number', 'Image URL', 'Review', 'Unnamed_13', 'Unnamed_14', 'AI Generated', 'From Form']


In [4]:
df.head()

,Days in Warehouse,Manufacturer Name,Category,General,Name,Lot Number,Notes,Quantity,Date of Expiration,Pallet Number,Box Number,Image URL,Review,Unnamed_13,Unnamed_14,AI Generated,From Form
0,0,"Medical Technology, Inc.",Misc Surgical,{medical device},{unidentified medical device},unspecified,extracted text from shipping label; general na...,1,,8,MISC 2,https://drive.google.com/open?id=1pJX0gctFKliG...,No review needed,,,,
1,0,"Cardinal Health 200, LLC",Misc Surgical,{sterile wipe/pad},{sterile wipe/pad},PT00108991-11,extracted text; generalName and name are guess...,100,,8,1,https://drive.google.com/open?id=1PKxKEEDGLAoL...,No review needed,,,,
2,0,Medline,Wound Care & Bandages,abdominal pad,EXTRA ABSORBENT ABDOMINAL PAD 8 x 10 IN,6052504001,Product name extracted from the prominent text...,1,,7,9,https://drive.google.com/open?id=10sbBWJce6VT0...,No review needed,,,,
3,0,Medline,Wound Care & Bandages,abdominal pad,EXTRA ABSORBENT ABDOMINAL PAD,6052509018,extracted text,5,,7,ER #1,https://drive.google.com/open?id=1h5Byf6a8L1sJ...,No review needed,,,,
4,0,Medline,Wound Care & Bandages,Abdominal Pad,EXTRA ABSORBENT ABDOMINAL PAD REF PRM21454 8 x...,6052508005,extracted text,3,,7,2,https://drive.google.com/open?id=1U691kT3o5Ez6...,No review needed,,,,
